# Robosuite Scripted Data Collection + SmolVLA Training & Evaluation

End-to-end pipeline for 6 robosuite manipulation tasks:

| # | Environment | Description | Target Demos |
|---|---|---|---|
| 1 | **Lift** | Pick up a cube and lift it | 50 |
| 2 | **Stack** | Pick red cube, stack on green cube | 50 |
| 3 | **PickPlaceSingle** | Pick object, place in target bin | 50 |
| 4 | **NutAssemblySquare** | Place square nut on square peg | 75 |
| 5 | **NutAssemblyRound** | Place round nut on round peg | 75 |
| 6 | **NutAssembly** | Both nuts on their pegs | 75 |

## Pipeline
1. **Setup** — Install deps, configure EGL rendering
2. **Scripted Policies** — Proper hover -> descend -> grip -> lift -> move -> place sequences
3. **Trial Runs** — 2 episodes per scenario with video (verify before full run)
4. **Full Collection** — HDF5 demos (50 each, 75 for nut assembly)
5. **Convert to LeRobot** — HDF5 -> LeRobot v3.0 format for SmolVLA training
6. **Train SmolVLA** — Fine-tune on collected demos
7. **Evaluate VLA** — Test trained model on all 6 envs with video + success reporting

**Requirements:** GPU runtime (Runtime -> Change runtime type -> T4 or A100 GPU)

---
## 1. System Setup & Dependencies

In [ ]:
%%bash
# Install system dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# Create NVIDIA EGL ICD config (Colab is missing this by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
# Install Python packages
# 1) Simulation + data
!pip install -q robosuite imageio[ffmpeg] matplotlib h5py Pillow pandas

# 2) LeRobot with SmolVLA support (for training + evaluation)
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"

# 3) Pin numpy (mujoco needs >=2.0; numba needs <2.1)
!pip install -q numpy==2.0.2

print("\nPackages installed. Restarting runtime to fix numpy C bindings...")
print("After restart, SKIP this cell and continue from the next section.")

import os
os.kill(os.getpid(), 9)

### After runtime restart -- continue from here

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

import numpy as np
import robosuite as suite
import imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import base64
import h5py
import time
import json
import torch

print(f"robosuite {robosuite.__version__}")
print(f"numpy {np.__version__}")
print(f"torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete.")

---
## 2. Scripted Policies

Each policy uses a **state-machine** approach with proper phases:
1. **Hover** -- move above the target object (open gripper)
2. **Fine Align** -- precise XY centering at intermediate height (for non-cube objects)
3. **Descend** -- lower onto the object at shape-specific grasp point
4. **Grasp** -- close gripper with shape-specific timing for firm hold
5. **Lift** -- raise the object
6. **Move** -- transport to target location (for place/stack tasks)
7. **Align Target** -- fine alignment above bin/peg before placement
8. **Place/Release** -- open gripper at target + retreat

**Shape-specific parameters** adapt each phase per object type:
| Object | Dimensions | Hover | Grasp Offset | Grip Wait | Key Adaptation |
|---|---|---|---|---|---|
| Cube | 5x5x5cm | 12cm | +0.3cm | 15 steps | Standard (baseline) |
| Milk | 4.5x4.5x14cm | 15cm | -2cm | 20 steps | Grasp below center |
| Bread | 10x5.5x4cm | 10cm | 0cm | 30 steps | Slow descent, long grip |
| Cereal | 6x2.5x17cm | 18cm | -3cm | 20 steps | Grasp well below center |
| Can | 5.6cm dia x 12cm | 15cm | -1cm | 30 steps | Tight alignment, slow |
| Nuts | ~7cm body x 1.8cm | 10cm | -0.3cm | 30 steps | Fine align, low descent |

Gripper convention: **+1 = close, -1 = open** (verified from robosuite Panda source).

In [ ]:
# ============================================================
# SCRIPTED POLICIES -- All 6 Scenarios
# ============================================================
#
# Gripper: +1 = close, -1 = open (robosuite PandaGripper)
#
# Action space (7D OSC_POSE for Panda):
#   [dx, dy, dz, dax, day, daz, gripper]
#   All values clipped to [-1, 1]
# ============================================================

TABLE_HEIGHT = 0.8
HOVER_DELTA_Z = 0.12
GRASP_XY_THRESH = 0.01
GRIP_WAIT_STEPS = 15
GAIN = 8.0

# ============================================================
# PANDA GRIPPER CLEARANCE
# ============================================================
GRIPPER_BODY_ABOVE_EEF = 0.055
GRIPPER_MAX_OPENING = 0.080

# ============================================================
# PEG DIMENSIONS (from pegs_arena.xml)
# ============================================================
# peg1 (square): box size=(0.016, 0.016, 0.1) at pos=(0.23, 0.1, 0.85)
#   → 32×32mm cross-section, peg top at z=0.85+0.1=0.95
# peg2 (round):  cylinder size=(0.02, 0.1) at pos=(0.23, -0.1, 0.85)
#   → 40mm diameter, peg top at z=0.95
PEG_HALF_HEIGHT = 0.10  # both pegs: geom half-size in z

# ============================================================
# OBJECT DIMENSIONS (from robosuite XML model files)
# ============================================================
OBJ_TOP_FROM_ORIGIN = {
    "Milk": 0.075,
    "Bread": 0.030,
    "Cereal": 0.030,
    "Can": 0.040,
}

OBJ_HALF_WIDTHS = {
    "Milk": (0.025, 0.025),
    "Bread": (0.03, 0.03),
    "Cereal": (0.04, 0.03),
    "Can": (0.025, 0.025),
}

def _safe_grasp_offset(obj_name, desired_offset=0.015):
    """Compute safe grasp z-offset ensuring gripper housing clears object top."""
    top_h = OBJ_TOP_FROM_ORIGIN.get(obj_name, 0.05)
    min_offset = top_h - GRIPPER_BODY_ABOVE_EEF + 0.020
    return max(desired_offset, min_offset)

PICK_PLACE_PARAMS = {
    "Milk": {
        "hover_z": 0.18,
        "grasp_z_offset": _safe_grasp_offset("Milk"),
        "descend_gain": 3.5,
        "grip_wait": 25,
        "xy_thresh": 0.012,
        "lift_clearance": 0.25,
    },
    "Bread": {
        "hover_z": 0.10,
        "grasp_z_offset": _safe_grasp_offset("Bread", desired_offset=-0.005),
        "descend_gain": 4.0,
        "grip_wait": 30,
        "xy_thresh": 0.008,
        "lift_clearance": 0.22,
    },
    "Cereal": {
        "hover_z": 0.18,
        "grasp_z_offset": _safe_grasp_offset("Cereal"),
        "descend_gain": 3.5,
        "grip_wait": 25,
        "xy_thresh": 0.010,
        "lift_clearance": 0.25,
    },
    "Can": {
        "hover_z": 0.14,
        "grasp_z_offset": _safe_grasp_offset("Can"),
        "descend_gain": 4.0,
        "grip_wait": 30,
        "xy_thresh": 0.008,
        "lift_clearance": 0.22,
    },
}

PICK_PLACE_DEFAULT = {
    "hover_z": HOVER_DELTA_Z,
    "grasp_z_offset": 0.015,
    "descend_gain": GAIN,
    "grip_wait": GRIP_WAIT_STEPS,
    "xy_thresh": GRASP_XY_THRESH,
    "lift_clearance": 0.20,
}

OBJ_BIN_INDEX = {"Milk": 0, "Bread": 1, "Cereal": 2, "Can": 3}

NUT_WALL_MID = {
    "SquareNut": 0.03325,
    "RoundNut": 0.04245,
}

NUT_PARAMS = {
    "SquareNut": {
        "hover_z": 0.10,
        "grasp_z_offset": 0.002,
        "descend_gain": 3.5,
        "grip_wait": 30,
        "xy_thresh": 0.006,
        "fine_z": 0.04,
        "fine_gain": 3.0,
        "peg_insert_depth": 0.015,  # lower nut 15mm below peg top
        "peg_gain": 2.5,
    },
    "RoundNut": {
        "hover_z": 0.10,
        "grasp_z_offset": 0.002,
        "descend_gain": 3.5,
        "grip_wait": 30,
        "xy_thresh": 0.006,
        "fine_z": 0.04,
        "fine_gain": 3.0,
        "peg_insert_depth": 0.015,
        "peg_gain": 2.5,
    },
}


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _quat_rotate(quat_xyzw, vec):
    """Rotate a 3D vector by quaternion in (x, y, z, w) format."""
    x, y, z, w = quat_xyzw
    qv = np.array([x, y, z])
    t = 2.0 * np.cross(qv, vec)
    return vec + w * t + np.cross(qv, t)


def _pick_place_jaw_align(obs, obj_name, eef_quat):
    """Compute daz to align gripper jaws with object's narrow side."""
    hx, hy = OBJ_HALF_WIDTHS.get(obj_name, (0.025, 0.025))
    if abs(hx - hy) < 0.005:
        return 0.0
    obj_quat = obs.get(f"{obj_name}_quat", np.array([0., 0., 0., 1.]))
    narrow_local = np.array([0., 1., 0.]) if hy < hx else np.array([1., 0., 0.])
    narrow_world = _quat_rotate(obj_quat, narrow_local)
    narrow_world[2] = 0.0
    n_norm = np.linalg.norm(narrow_world)
    if n_norm < 1e-6:
        return 0.0
    narrow_world /= n_norm
    jaw_dir = _quat_rotate(eef_quat, np.array([0., 1., 0.]))
    jaw_dir[2] = 0.0
    j_norm = np.linalg.norm(jaw_dir)
    if j_norm < 1e-6:
        return 0.0
    jaw_dir /= j_norm
    sin_error = jaw_dir[0] * narrow_world[1] - jaw_dir[1] * narrow_world[0]
    return -sin_error * 4.0


def _nut_jaw_grasp_target(nut_pos, eef_quat, wall_mid):
    """Compute grasp target so gripper jaws straddle the nut wall."""
    local_y = np.array([0., 1., 0.])
    jaw_dir = _quat_rotate(eef_quat, local_y)
    jaw_dir[2] = 0.0
    norm = np.linalg.norm(jaw_dir)
    if norm < 1e-6:
        jaw_dir = np.array([0., 1., 0.])
    else:
        jaw_dir = jaw_dir / norm
    target = nut_pos.copy()
    target[0] += wall_mid * jaw_dir[0]
    target[1] += wall_mid * jaw_dir[1]
    return target


def _compute_jaw_dir(eef_quat):
    """Compute the gripper jaw direction projected onto XY plane."""
    local_y = np.array([0., 1., 0.])
    jaw_dir = _quat_rotate(eef_quat, local_y)
    jaw_dir[2] = 0.0
    norm = np.linalg.norm(jaw_dir)
    if norm < 1e-6:
        jaw_dir = np.array([0., 1., 0.])
    else:
        jaw_dir = jaw_dir / norm
    return jaw_dir


def _nut_grasp_from_jaw_dir(nut_pos, jaw_dir, wall_mid):
    """Compute grasp target using a pre-locked jaw direction."""
    target = nut_pos.copy()
    target[0] += wall_mid * jaw_dir[0]
    target[1] += wall_mid * jaw_dir[1]
    return target


def _nut_z_angle(nut_quat):
    """Extract z-rotation (yaw) from quaternion (x, y, z, w)."""
    x, y, z, w = nut_quat
    return np.arctan2(2.0 * (w * z + x * y), 1.0 - 2.0 * (y * y + z * z))


def _square_nut_align_error(nut_quat):
    """Compute z-rotation error for square nut → nearest 90° alignment."""
    yaw = _nut_z_angle(nut_quat)
    target = round(yaw / (np.pi / 2.0)) * (np.pi / 2.0)
    error = target - yaw
    while error > np.pi / 4.0:
        error -= np.pi / 2.0
    while error < -np.pi / 4.0:
        error += np.pi / 2.0
    return error


# ============================================================
# LiftPolicy -- UNCHANGED
# ============================================================
class LiftPolicy:
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        cube = obs["cube_pos"]
        action = np.zeros(7)
        if self.phase == "hover":
            target = cube.copy(); target[2] += HOVER_DELTA_Z
            delta = target - ee; action[:3] = delta * GAIN; action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend"
        elif self.phase == "descend":
            target = cube.copy(); target[2] += 0.003
            delta = target - ee; action[:3] = delta * GAIN; action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip"; self.grip_counter = 0
        elif self.phase == "grip":
            target = cube.copy(); target[2] += 0.003
            delta = target - ee; action[:3] = delta * GAIN * 0.5; action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS: self.phase = "lift"
        elif self.phase == "lift":
            action[2] = 1.0; action[6] = 1
        return np.clip(action, -1, 1)


# ============================================================
# StackPolicy -- UNCHANGED
# ============================================================
class StackPolicy:
    def __init__(self):
        self.phase = "hover_A"; self.grip_counter = 0; self.release_counter = 0
    def reset(self):
        self.phase = "hover_A"; self.grip_counter = 0; self.release_counter = 0
    def __call__(self, obs):
        ee = obs["robot0_eef_pos"]
        cubeA = obs["cubeA_pos"]; cubeB = obs["cubeB_pos"]
        action = np.zeros(7)
        if self.phase == "hover_A":
            target = cubeA.copy(); target[2] += HOVER_DELTA_Z
            delta = target - ee; action[:3] = delta * GAIN; action[6] = -1
            if np.linalg.norm(delta[:2]) < GRASP_XY_THRESH and abs(delta[2]) < 0.02:
                self.phase = "descend_A"
        elif self.phase == "descend_A":
            target = cubeA.copy(); target[2] += 0.003
            delta = target - ee; action[:3] = delta * GAIN; action[6] = -1
            if np.linalg.norm(delta) < 0.01:
                self.phase = "grip_A"; self.grip_counter = 0
        elif self.phase == "grip_A":
            target = cubeA.copy(); target[2] += 0.003
            delta = target - ee; action[:3] = delta * GAIN * 0.5; action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= GRIP_WAIT_STEPS: self.phase = "lift_A"
        elif self.phase == "lift_A":
            action[2] = 1.0; action[6] = 1
            if ee[2] > cubeB[2] + 0.15: self.phase = "hover_B"
        elif self.phase == "hover_B":
            target = cubeB.copy(); target[2] += 0.10
            delta = target - ee; action[:3] = delta * GAIN * 0.6; action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.015 and abs(delta[2]) < 0.02:
                self.phase = "descend_B"
        elif self.phase == "descend_B":
            target = cubeB.copy(); target[2] += 0.05
            delta = target - ee; action[:3] = delta * GAIN * 0.5; action[6] = 1
            if np.linalg.norm(delta) < 0.015:
                self.phase = "release"; self.release_counter = 0
        elif self.phase == "release":
            action[6] = -1; self.release_counter += 1
            if self.release_counter > 10: action[2] = 1.0
        return np.clip(action, -1, 1)


# ============================================================
# PickPlaceSinglePolicy -- JAW ROTATION + DESCENT TIMEOUT
# ============================================================
class PickPlaceSinglePolicy:
    """PickPlaceSingle: pick object -> place in CORRECT bin compartment.
    Jaw rotation for wide objects. Descent timeout prevents stuck jaws.
    """
    OBJ_NAMES = ["Milk", "Bread", "Cereal", "Can"]

    def __init__(self):
        self.phase = "hover"; self.grip_counter = 0
        self.release_counter = 0; self.descend_counter = 0
        self.active_obj = None

    def reset(self):
        self.phase = "hover"; self.grip_counter = 0
        self.release_counter = 0; self.descend_counter = 0
        self.active_obj = None

    def _find_object(self, obs):
        if self.active_obj is not None: return self.active_obj
        for name in self.OBJ_NAMES:
            if f"{name}_pos" in obs:
                self.active_obj = name; return name
        return None

    def _params(self):
        if self.active_obj and self.active_obj in PICK_PLACE_PARAMS:
            return PICK_PLACE_PARAMS[self.active_obj]
        return PICK_PLACE_DEFAULT

    def __call__(self, obs, bin_compartments=None):
        ee = obs["robot0_eef_pos"]
        eef_quat = obs.get("robot0_eef_quat", np.array([0., 0., 0., 1.]))
        obj_name = self._find_object(obs)
        if obj_name is None: return np.zeros(7)
        obj_pos = obs[f"{obj_name}_pos"]
        if bin_compartments is not None and obj_name in OBJ_BIN_INDEX:
            bin_target = bin_compartments[OBJ_BIN_INDEX[obj_name]].copy()
        else:
            bin_target = np.array([0.1, 0.28, 0.8])
        p = self._params()
        action = np.zeros(7)
        daz_align = _pick_place_jaw_align(obs, obj_name, eef_quat)

        if self.phase == "hover":
            target = obj_pos.copy(); target[2] += p["hover_z"]
            delta = target - ee; action[:3] = delta * GAIN
            action[5] = np.clip(daz_align, -1, 1); action[6] = -1
            if np.linalg.norm(delta[:2]) < p["xy_thresh"] and abs(delta[2]) < 0.02:
                self.phase = "fine_align"
        elif self.phase == "fine_align":
            target = obj_pos.copy(); target[2] += p["hover_z"] * 0.4
            delta = target - ee; action[:3] = delta * p["descend_gain"] * 0.5
            action[5] = np.clip(daz_align, -1, 1); action[6] = -1
            if np.linalg.norm(delta[:2]) < p["xy_thresh"] * 0.7 and abs(delta[2]) < 0.015:
                self.phase = "descend"; self.descend_counter = 0
        elif self.phase == "descend":
            target = obj_pos.copy(); target[2] += p["grasp_z_offset"]
            delta = target - ee; action[:3] = delta * p["descend_gain"]
            action[5] = np.clip(daz_align * 0.5, -0.5, 0.5); action[6] = -1
            self.descend_counter += 1
            if np.linalg.norm(delta) < 0.01 or self.descend_counter > 50:
                self.phase = "grip"; self.grip_counter = 0
        elif self.phase == "grip":
            target = obj_pos.copy(); target[2] += p["grasp_z_offset"]
            delta = target - ee; action[:3] = delta * p["descend_gain"] * 0.3
            action[6] = 1; self.grip_counter += 1
            if self.grip_counter >= p["grip_wait"]: self.phase = "lift"
        elif self.phase == "lift":
            action[2] = 1.0; action[6] = 1
            if ee[2] > bin_target[2] + p["lift_clearance"]:
                self.phase = "move_to_bin"
        elif self.phase == "move_to_bin":
            target = bin_target.copy(); target[2] = ee[2]
            delta = target - ee; action[:3] = delta * GAIN * 0.5; action[6] = 1
            if np.linalg.norm(delta[:2]) < 0.03: self.phase = "lower_to_bin"
        elif self.phase == "lower_to_bin":
            target = bin_target.copy(); target[2] += 0.10
            delta = target - ee; action[:3] = delta * GAIN * 0.4; action[6] = 1
            if abs(delta[2]) < 0.02:
                self.phase = "release"; self.release_counter = 0
        elif self.phase == "release":
            action[6] = -1; self.release_counter += 1
            if self.release_counter > 10: action[2] = 1.0
        return np.clip(action, -1, 1)


# ============================================================
# NutAssemblySquarePolicy -- NUT-CENTERED PLACEMENT + ROTATION
# ============================================================
class NutAssemblySquarePolicy:
    """NutAssemblySquare: pick square nut -> rotate -> center hole over peg -> slide on.

    CRITICAL: During placement, we control based on NUT CENTER position
    (from obs), NOT EEF position. The nut is gripped at the wall (~33mm
    from hole center), so EEF is offset from nut center. We drive the
    nut's hole center onto the peg center using closed-loop control on
    SquareNut_pos.

    Peg top z = peg_body_z + 0.10 (half-height). We lower the nut 15mm
    below peg top so it engages, then release. Gravity slides it down.
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.lower_counter = 0
        self.hover_counter = 0
        self.fine_counter = 0
        self.locked_jaw_dir = None
        self.p = NUT_PARAMS["SquareNut"]
        self.wall_mid = NUT_WALL_MID["SquareNut"]

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.lower_counter = 0
        self.hover_counter = 0
        self.fine_counter = 0
        self.locked_jaw_dir = None

    def __call__(self, obs, peg_pos=None):
        ee = obs["robot0_eef_pos"]
        eef_quat = obs.get("robot0_eef_quat", np.array([0, 0, 0, 1]))
        nut_pos = obs["SquareNut_pos"]
        action = np.zeros(7)
        p = self.p
        if peg_pos is None:
            peg_pos = np.array([0.23, 0.1, 0.85])

        # Lock jaw direction on first call to prevent moving-target oscillation
        if self.locked_jaw_dir is None:
            self.locked_jaw_dir = _compute_jaw_dir(eef_quat)
        grasp_xy = _nut_grasp_from_jaw_dir(nut_pos, self.locked_jaw_dir, self.wall_mid)
        grasp_target = grasp_xy.copy()
        grasp_target[2] = nut_pos[2] + p["grasp_z_offset"]

        if self.phase == "hover":
            target = grasp_xy.copy()
            target[2] = nut_pos[2] + p["hover_z"]
            delta = target - ee
            action[:3] = delta * GAIN; action[6] = -1
            self.hover_counter += 1
            if (np.linalg.norm(delta[:2]) < p["xy_thresh"] and abs(delta[2]) < 0.02) or self.hover_counter > 100:
                self.phase = "fine_align"; self.fine_counter = 0

        elif self.phase == "fine_align":
            target = grasp_xy.copy()
            target[2] = nut_pos[2] + p["fine_z"]
            delta = target - ee
            action[:3] = delta * p["fine_gain"]; action[6] = -1
            self.fine_counter += 1
            if (np.linalg.norm(delta[:2]) < 0.004 and abs(delta[2]) < 0.01) or self.fine_counter > 80:
                self.phase = "descend"; self.descend_counter = 0

        elif self.phase == "descend":
            delta = grasp_target - ee
            action[:3] = delta * p["descend_gain"]; action[6] = -1
            self.descend_counter = getattr(self, 'descend_counter', 0) + 1
            if np.linalg.norm(delta) < 0.008 or self.descend_counter > 60:
                self.phase = "grip"; self.grip_counter = 0

        elif self.phase == "grip":
            delta = grasp_target - ee
            action[:3] = delta * p["descend_gain"] * 0.3; action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= p["grip_wait"]: self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0; action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "rotate_align"

        elif self.phase == "rotate_align":
            # Rotate nut to align square hole with axis-aligned square peg
            action[6] = 1; action[2] = 0.1
            nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
            error = _square_nut_align_error(nut_quat)
            action[5] = np.clip(error * 3.0, -1, 1)
            if abs(error) < 0.10:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            # Drive NUT CENTER over PEG CENTER (closed-loop on nut_pos)
            nut_xy_err = nut_pos[:2] - peg_pos[:2]
            action[0] = -nut_xy_err[0] * GAIN * 0.5
            action[1] = -nut_xy_err[1] * GAIN * 0.5
            action[2] = 0.1  # maintain height
            action[6] = 1
            # Maintain rotation
            nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
            rot_err = _square_nut_align_error(nut_quat)
            action[5] = np.clip(rot_err * 2.0, -0.5, 0.5)
            if np.linalg.norm(nut_xy_err) < 0.012:
                self.phase = "align_peg"

        elif self.phase == "align_peg":
            # Fine XY centering of nut hole over peg + rotation
            nut_xy_err = nut_pos[:2] - peg_pos[:2]
            action[0] = -nut_xy_err[0] * p["peg_gain"] * 2.0
            action[1] = -nut_xy_err[1] * p["peg_gain"] * 2.0
            action[6] = 1
            nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
            rot_err = _square_nut_align_error(nut_quat)
            action[5] = np.clip(rot_err * 2.0, -0.5, 0.5)
            if np.linalg.norm(nut_xy_err) < 0.006 and abs(rot_err) < 0.12:
                self.phase = "lower_to_peg"; self.lower_counter = 0

        elif self.phase == "lower_to_peg":
            # Lower nut onto peg: target nut center at peg_top - insert_depth
            peg_top_z = peg_pos[2] + PEG_HALF_HEIGHT
            target_nut_z = peg_top_z - p["peg_insert_depth"]
            # EEF is above nut center by ~grasp_z_offset
            target_eef_z = target_nut_z + p["grasp_z_offset"]
            # Maintain XY centering on nut hole
            nut_xy_err = nut_pos[:2] - peg_pos[:2]
            action[0] = -nut_xy_err[0] * p["peg_gain"] * 2.0
            action[1] = -nut_xy_err[1] * p["peg_gain"] * 2.0
            action[2] = (target_eef_z - ee[2]) * p["peg_gain"]
            action[6] = 1
            # Maintain rotation
            nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
            rot_err = _square_nut_align_error(nut_quat)
            action[5] = np.clip(rot_err * 2.0, -0.5, 0.5)
            self.lower_counter += 1
            if abs(ee[2] - target_eef_z) < 0.015 or self.lower_counter > 60:
                self.phase = "release"; self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1; self.release_counter += 1
            if self.release_counter > 10: action[2] = 1.0

        return np.clip(action, -1, 1)


# ============================================================
# NutAssemblyRoundPolicy -- NUT-CENTERED PLACEMENT (no rotation)
# ============================================================
class NutAssemblyRoundPolicy:
    """NutAssemblyRound: pick round nut -> center hole over round peg -> slide on.
    Round peg is cylindrical so no rotation needed. Same nut-centered
    placement as square nut.
    """
    def __init__(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.lower_counter = 0
        self.hover_counter = 0
        self.fine_counter = 0
        self.locked_jaw_dir = None
        self.p = NUT_PARAMS["RoundNut"]
        self.wall_mid = NUT_WALL_MID["RoundNut"]

    def reset(self):
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.lower_counter = 0
        self.hover_counter = 0
        self.fine_counter = 0
        self.locked_jaw_dir = None

    def __call__(self, obs, peg_pos=None):
        ee = obs["robot0_eef_pos"]
        eef_quat = obs.get("robot0_eef_quat", np.array([0, 0, 0, 1]))
        nut_pos = obs["RoundNut_pos"]
        action = np.zeros(7)
        p = self.p
        if peg_pos is None:
            peg_pos = np.array([0.23, -0.1, 0.85])

        # Lock jaw direction on first call to prevent moving-target oscillation
        if self.locked_jaw_dir is None:
            self.locked_jaw_dir = _compute_jaw_dir(eef_quat)
        grasp_xy = _nut_grasp_from_jaw_dir(nut_pos, self.locked_jaw_dir, self.wall_mid)
        grasp_target = grasp_xy.copy()
        grasp_target[2] = nut_pos[2] + p["grasp_z_offset"]

        if self.phase == "hover":
            target = grasp_xy.copy()
            target[2] = nut_pos[2] + p["hover_z"]
            delta = target - ee
            action[:3] = delta * GAIN; action[6] = -1
            self.hover_counter += 1
            if (np.linalg.norm(delta[:2]) < p["xy_thresh"] and abs(delta[2]) < 0.02) or self.hover_counter > 100:
                self.phase = "fine_align"; self.fine_counter = 0

        elif self.phase == "fine_align":
            target = grasp_xy.copy()
            target[2] = nut_pos[2] + p["fine_z"]
            delta = target - ee
            action[:3] = delta * p["fine_gain"]; action[6] = -1
            self.fine_counter += 1
            if (np.linalg.norm(delta[:2]) < 0.004 and abs(delta[2]) < 0.01) or self.fine_counter > 80:
                self.phase = "descend"; self.descend_counter = 0

        elif self.phase == "descend":
            delta = grasp_target - ee
            action[:3] = delta * p["descend_gain"]; action[6] = -1
            self.descend_counter = getattr(self, 'descend_counter', 0) + 1
            if np.linalg.norm(delta) < 0.008 or self.descend_counter > 60:
                self.phase = "grip"; self.grip_counter = 0

        elif self.phase == "grip":
            delta = grasp_target - ee
            action[:3] = delta * p["descend_gain"] * 0.3; action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= p["grip_wait"]: self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0; action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            # Drive NUT CENTER over PEG CENTER
            nut_xy_err = nut_pos[:2] - peg_pos[:2]
            action[0] = -nut_xy_err[0] * GAIN * 0.5
            action[1] = -nut_xy_err[1] * GAIN * 0.5
            action[2] = 0.1; action[6] = 1
            if np.linalg.norm(nut_xy_err) < 0.012:
                self.phase = "align_peg"

        elif self.phase == "align_peg":
            nut_xy_err = nut_pos[:2] - peg_pos[:2]
            action[0] = -nut_xy_err[0] * p["peg_gain"] * 2.0
            action[1] = -nut_xy_err[1] * p["peg_gain"] * 2.0
            action[6] = 1
            if np.linalg.norm(nut_xy_err) < 0.006:
                self.phase = "lower_to_peg"; self.lower_counter = 0

        elif self.phase == "lower_to_peg":
            peg_top_z = peg_pos[2] + PEG_HALF_HEIGHT
            target_nut_z = peg_top_z - p["peg_insert_depth"]
            target_eef_z = target_nut_z + p["grasp_z_offset"]
            nut_xy_err = nut_pos[:2] - peg_pos[:2]
            action[0] = -nut_xy_err[0] * p["peg_gain"] * 2.0
            action[1] = -nut_xy_err[1] * p["peg_gain"] * 2.0
            action[2] = (target_eef_z - ee[2]) * p["peg_gain"]
            action[6] = 1
            self.lower_counter += 1
            if abs(ee[2] - target_eef_z) < 0.015 or self.lower_counter > 60:
                self.phase = "release"; self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1; self.release_counter += 1
            if self.release_counter > 10: action[2] = 1.0

        return np.clip(action, -1, 1)


# ============================================================
# NutAssemblyFullPolicy -- BOTH NUTS
# ============================================================
class NutAssemblyFullPolicy:
    """NutAssembly (both): square nut -> peg1, then round nut -> peg2.
    Uses nut-centered placement for both. Square nut gets rotation alignment.
    """
    def __init__(self):
        self.current_nut = "square"
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.retreat_counter = 0
        self.lower_counter = 0
        self.hover_counter = 0
        self.fine_counter = 0
        self.locked_jaw_dir = None

    def reset(self):
        self.current_nut = "square"
        self.phase = "hover"
        self.grip_counter = 0
        self.release_counter = 0
        self.retreat_counter = 0
        self.lower_counter = 0
        self.hover_counter = 0
        self.fine_counter = 0
        self.locked_jaw_dir = None

    def __call__(self, obs, peg1_pos=None, peg2_pos=None):
        ee = obs["robot0_eef_pos"]
        eef_quat = obs.get("robot0_eef_quat", np.array([0, 0, 0, 1]))
        action = np.zeros(7)

        if self.current_nut == "square":
            nut_pos = obs["SquareNut_pos"]
            peg = peg1_pos if peg1_pos is not None else np.array([0.23, 0.1, 0.85])
            p = NUT_PARAMS["SquareNut"]
            wall_mid = NUT_WALL_MID["SquareNut"]
        else:
            nut_pos = obs["RoundNut_pos"]
            peg = peg2_pos if peg2_pos is not None else np.array([0.23, -0.1, 0.85])
            p = NUT_PARAMS["RoundNut"]
            wall_mid = NUT_WALL_MID["RoundNut"]

        # Lock jaw direction on first call to prevent moving-target oscillation
        if self.locked_jaw_dir is None:
            self.locked_jaw_dir = _compute_jaw_dir(eef_quat)
        grasp_xy = _nut_grasp_from_jaw_dir(nut_pos, self.locked_jaw_dir, wall_mid)
        grasp_target = grasp_xy.copy()
        grasp_target[2] = nut_pos[2] + p["grasp_z_offset"]

        if self.phase == "hover":
            target = grasp_xy.copy()
            target[2] = nut_pos[2] + p["hover_z"]
            delta = target - ee
            action[:3] = delta * GAIN; action[6] = -1
            self.hover_counter += 1
            if (np.linalg.norm(delta[:2]) < p["xy_thresh"] and abs(delta[2]) < 0.02) or self.hover_counter > 100:
                self.phase = "fine_align"; self.fine_counter = 0

        elif self.phase == "fine_align":
            target = grasp_xy.copy()
            target[2] = nut_pos[2] + p["fine_z"]
            delta = target - ee
            action[:3] = delta * p["fine_gain"]; action[6] = -1
            self.fine_counter += 1
            if (np.linalg.norm(delta[:2]) < 0.004 and abs(delta[2]) < 0.01) or self.fine_counter > 80:
                self.phase = "descend"; self.descend_counter = 0

        elif self.phase == "descend":
            delta = grasp_target - ee
            action[:3] = delta * p["descend_gain"]; action[6] = -1
            self.descend_counter = getattr(self, 'descend_counter', 0) + 1
            if np.linalg.norm(delta) < 0.008 or self.descend_counter > 60:
                self.phase = "grip"; self.grip_counter = 0

        elif self.phase == "grip":
            delta = grasp_target - ee
            action[:3] = delta * p["descend_gain"] * 0.3; action[6] = 1
            self.grip_counter += 1
            if self.grip_counter >= p["grip_wait"]: self.phase = "lift"

        elif self.phase == "lift":
            action[2] = 1.0; action[6] = 1
            if ee[2] > TABLE_HEIGHT + 0.20:
                if self.current_nut == "square":
                    self.phase = "rotate_align"
                else:
                    self.phase = "move_to_peg"

        elif self.phase == "rotate_align":
            action[6] = 1; action[2] = 0.1
            nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
            error = _square_nut_align_error(nut_quat)
            action[5] = np.clip(error * 3.0, -1, 1)
            if abs(error) < 0.10: self.phase = "move_to_peg"

        elif self.phase == "move_to_peg":
            # Drive NUT CENTER over PEG CENTER
            nut_xy_err = nut_pos[:2] - peg[:2]
            action[0] = -nut_xy_err[0] * GAIN * 0.5
            action[1] = -nut_xy_err[1] * GAIN * 0.5
            action[2] = 0.1; action[6] = 1
            if self.current_nut == "square":
                nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
                rot_err = _square_nut_align_error(nut_quat)
                action[5] = np.clip(rot_err * 2.0, -0.5, 0.5)
            if np.linalg.norm(nut_xy_err) < 0.012:
                self.phase = "align_peg"

        elif self.phase == "align_peg":
            nut_xy_err = nut_pos[:2] - peg[:2]
            action[0] = -nut_xy_err[0] * p["peg_gain"] * 2.0
            action[1] = -nut_xy_err[1] * p["peg_gain"] * 2.0
            action[6] = 1
            if self.current_nut == "square":
                nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
                rot_err = _square_nut_align_error(nut_quat)
                action[5] = np.clip(rot_err * 2.0, -0.5, 0.5)
                aligned = np.linalg.norm(nut_xy_err) < 0.006 and abs(rot_err) < 0.12
            else:
                aligned = np.linalg.norm(nut_xy_err) < 0.006
            if aligned:
                self.phase = "lower_to_peg"; self.lower_counter = 0

        elif self.phase == "lower_to_peg":
            peg_top_z = peg[2] + PEG_HALF_HEIGHT
            target_nut_z = peg_top_z - p["peg_insert_depth"]
            target_eef_z = target_nut_z + p["grasp_z_offset"]
            nut_xy_err = nut_pos[:2] - peg[:2]
            action[0] = -nut_xy_err[0] * p["peg_gain"] * 2.0
            action[1] = -nut_xy_err[1] * p["peg_gain"] * 2.0
            action[2] = (target_eef_z - ee[2]) * p["peg_gain"]
            action[6] = 1
            if self.current_nut == "square":
                nut_quat = obs.get("SquareNut_quat", np.array([0, 0, 0, 1]))
                rot_err = _square_nut_align_error(nut_quat)
                action[5] = np.clip(rot_err * 2.0, -0.5, 0.5)
            self.lower_counter += 1
            if abs(ee[2] - target_eef_z) < 0.015 or self.lower_counter > 60:
                self.phase = "release"; self.release_counter = 0

        elif self.phase == "release":
            action[6] = -1; self.release_counter += 1
            if self.release_counter > 15:
                self.phase = "retreat"; self.retreat_counter = 0

        elif self.phase == "retreat":
            action[2] = 1.0; action[6] = -1
            self.retreat_counter += 1
            if self.retreat_counter > 20:
                if self.current_nut == "square":
                    self.current_nut = "round"
                    self.phase = "hover"
                    self.grip_counter = 0
                    self.release_counter = 0
                    self.retreat_counter = 0
                    self.lower_counter = 0
                    self.hover_counter = 0
                    self.fine_counter = 0
                    self.locked_jaw_dir = None
                else:
                    self.phase = "done"

        elif self.phase == "done":
            pass

        return np.clip(action, -1, 1)


# Print computed grasp offsets for verification
for name in ["Milk", "Bread", "Cereal", "Can"]:
    offset = _safe_grasp_offset(name)
    top = OBJ_TOP_FROM_ORIGIN[name]
    clearance = offset + GRIPPER_BODY_ABOVE_EEF - top
    hx, hy = OBJ_HALF_WIDTHS[name]
    needs_rotate = "YES" if abs(hx - hy) >= 0.005 else "no"
    print(f"  {name:7s}: top={top:.3f}m, offset={offset:.3f}m, "
          f"clearance={clearance*1000:.0f}mm, "
          f"size={hx*2000:.0f}x{hy*2000:.0f}mm, rotate={needs_rotate}")

print(f"\nNut placement: closed-loop on nut_pos (not EEF)")
print(f"  Peg top z = peg_body_z + {PEG_HALF_HEIGHT}m")
print(f"  Square nut: rotate to nearest 90° + center hole over 32x32mm peg")
print(f"  Round nut: center hole over 40mm dia peg (no rotation needed)")
print(f"\nAll 6 scripted policies defined.")

---
## 3. Environment Config & Episode Runner

In [ ]:
# ============================================================
# Task descriptions for each scenario (used by SmolVLA)
# ============================================================
TASK_DESCRIPTIONS = {
    "Lift": "Pick up the cube and lift it off the table",
    "Stack": "Pick up the red cube and stack it on top of the green cube",
    "PickPlaceSingle": "Pick up the object and place it in the bin",
    "NutAssemblySquare": "Pick up the square nut and place it on the square peg",
    "NutAssemblyRound": "Pick up the round nut and place it on the round peg",
    "NutAssembly": "Assemble both the square nut and round nut onto their respective pegs",
}

SCENARIOS = {
    "Lift": {
        "env_kwargs": dict(
            env_name="Lift", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=400,
        ),
        "policy_class": LiftPolicy, "max_steps": 400, "target_demos": 50,
    },
    "Stack": {
        "env_kwargs": dict(
            env_name="Stack", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=500,
        ),
        "policy_class": StackPolicy, "max_steps": 500, "target_demos": 50,
    },
    "PickPlaceSingle": {
        "env_kwargs": dict(
            env_name="PickPlaceSingle", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=500,
        ),
        "policy_class": PickPlaceSinglePolicy, "max_steps": 500, "target_demos": 50,
    },
    "NutAssemblySquare": {
        "env_kwargs": dict(
            env_name="NutAssemblySquare", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=600,
        ),
        "policy_class": NutAssemblySquarePolicy, "max_steps": 600, "target_demos": 75,
    },
    "NutAssemblyRound": {
        "env_kwargs": dict(
            env_name="NutAssemblyRound", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, horizon=600,
        ),
        "policy_class": NutAssemblyRoundPolicy, "max_steps": 600, "target_demos": 75,
    },
    "NutAssembly": {
        "env_kwargs": dict(
            env_name="NutAssembly", robots="Panda",
            has_renderer=False, has_offscreen_renderer=True,
            use_camera_obs=True, use_object_obs=True,
            camera_names=["agentview", "robot0_eye_in_hand"],
            camera_heights=256, camera_widths=256,
            reward_shaping=True, single_object_mode=0, horizon=800,
        ),
        "policy_class": NutAssemblyFullPolicy, "max_steps": 800, "target_demos": 75,
    },
}


def _get_target_positions(env, scenario_name):
    """Read target positions (bins, pegs) from env for each scenario.

    PickPlaceSingle: computes 2x2 bin compartment grid from bin2_pos + bin_size.
      Each object type (Milk/Bread/Cereal/Can) maps to its own compartment.
    NutAssembly*: reads peg body positions from MuJoCo sim data.
    """
    if scenario_name == "PickPlaceSingle":
        # Compute 2x2 bin compartment grid from bin2 position and size
        bin2_pos = getattr(env, 'bin2_pos', np.array([0.1, 0.28, 0.8])).copy()
        bin_size = (0.4, 0.5)  # from robosuite PickPlace arena
        compartments = np.zeros((4, 3))
        # Index layout (matching OBJ_BIN_INDEX):
        #   0=Milk  (left,  near)    1=Bread   (right, near)
        #   2=Cereal(left,  far)     3=Can     (right, far)
        for i in range(4):
            bx = bin2_pos[0] - bin_size[0] / 2.0 + bin_size[0] / 4.0
            by = bin2_pos[1] - bin_size[1] / 2.0 + bin_size[1] / 4.0
            if i % 2 == 1:    # right column (Bread, Can)
                bx += bin_size[0] / 2.0
            if i >= 2:         # far row (Cereal, Can)
                by += bin_size[1] / 2.0
            compartments[i] = [bx, by, bin2_pos[2]]
        return {"bin_compartments": compartments}
    elif scenario_name == "NutAssemblySquare":
        return {"peg_pos": env.sim.data.body_xpos[env.peg1_body_id].copy()}
    elif scenario_name == "NutAssemblyRound":
        return {"peg_pos": env.sim.data.body_xpos[env.peg2_body_id].copy()}
    elif scenario_name == "NutAssembly":
        return {
            "peg1_pos": env.sim.data.body_xpos[env.peg1_body_id].copy(),
            "peg2_pos": env.sim.data.body_xpos[env.peg2_body_id].copy(),
        }
    return {}


def run_episode(env, policy, scenario_name, max_steps, record_video=False):
    """Run one episode with scripted policy.
    Returns dict: success, total_reward, steps, frames, actions, observations.
    Success is checked via env._check_success() (ground truth).
    """
    obs = env.reset()
    policy.reset()

    frames, actions_buf, obs_buf = [], [], []
    total_reward = 0.0
    success = False

    for step in range(max_steps):
        # Call policy with target positions (bin compartments for PickPlace, pegs for NutAssembly)
        target_kwargs = _get_target_positions(env, scenario_name)
        action = policy(obs, **target_kwargs)

        actions_buf.append(action.copy())

        # Store observation data for HDF5
        obs_entry = {}
        for key in ["agentview_image", "robot0_eye_in_hand_image",
                    "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]:
            if key in obs:
                obs_entry[key] = obs[key].copy()
        obs_buf.append(obs_entry)

        if record_video and "agentview_image" in obs:
            frames.append(np.flip(obs["agentview_image"], axis=0).copy())

        obs, reward, done, info = env.step(action)
        total_reward += reward

        if env._check_success():
            success = True

        if done:
            break

    if record_video and "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0).copy())

    return {
        "success": success,
        "total_reward": total_reward,
        "steps": len(actions_buf),
        "frames": frames,
        "actions": actions_buf,
        "observations": obs_buf,
    }


def save_video_file(frames, path, fps=20):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    writer = imageio.get_writer(path, fps=fps)
    for frame in frames:
        writer.append_data(frame)
    writer.close()


def show_video_inline(path):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="512">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
        f'</video>'
    ))


print(f"Configured {len(SCENARIOS)} scenarios:")
for name, cfg in SCENARIOS.items():
    print(f"  {name}: max_steps={cfg['max_steps']}, target={cfg['target_demos']}")

---
## 4. TRIAL RUNS -- 2 Episodes Per Scenario (with Video)

Run 2 episodes per scenario with video. **Check each video before full collection.**
Each episode prints SUCCESS or FAIL with color-coded headers.

In [ ]:
TRIAL_EPISODES = 2
trial_results = {}

for scenario_name, cfg in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"  TRIAL: {scenario_name} ({TRIAL_EPISODES} episodes)")
    print(f"{'='*60}")

    env = suite.make(**cfg["env_kwargs"])
    policy = cfg["policy_class"]()
    results = []

    for ep in range(TRIAL_EPISODES):
        np.random.seed(42 + ep)
        result = run_episode(
            env, policy, scenario_name,
            max_steps=cfg["max_steps"],
            record_video=True,
        )
        results.append(result)

        status = "SUCCESS" if result["success"] else "FAIL"
        color = "green" if result["success"] else "red"
        print(f"  Episode {ep+1}: {status}  "
              f"(reward={result['total_reward']:.2f}, steps={result['steps']})")

        video_path = f"trial_videos/{scenario_name}_ep{ep+1}.mp4"
        if result["frames"]:
            save_video_file(result["frames"], video_path)
            display(HTML(
                f'<h4 style="color: {color}; border: 2px solid {color}; '
                f'padding: 4px 8px; display: inline-block;">'
                f'{scenario_name} | Episode {ep+1} | {status}</h4>'
            ))
            show_video_inline(video_path)

    env.close()

    n_success = sum(1 for r in results if r["success"])
    trial_results[scenario_name] = {
        "success": n_success,
        "total": TRIAL_EPISODES,
        "rate": n_success / TRIAL_EPISODES,
    }

# ============================================================
# TRIAL SUMMARY TABLE
# ============================================================
print(f"\n\n{'='*60}")
print(f"  TRIAL SUMMARY")
print(f"{'='*60}")
print(f"  {'Scenario':<25s} {'Result':>10s} {'Rate':>8s} {'Status':>8s}")
print(f"  {'-'*55}")
for name, r in trial_results.items():
    status = "PASS" if r["success"] > 0 else "FAIL"
    print(f"  {name:<25s} {r['success']}/{r['total']:>5d} {r['rate']:>7.0%} {status:>8s}")

all_pass = all(r["success"] > 0 for r in trial_results.values())
print(f"\n  {'ALL PASSED -- safe to proceed' if all_pass else 'SOME FAILED -- review videos above'}")

---
## 5. FULL DATA COLLECTION

Collect successful demos to HDF5. Only saves episodes where `env._check_success()` is True.
- Lift, Stack, PickPlaceSingle: **50** successful demos each
- NutAssemblySquare, NutAssemblyRound, NutAssembly: **75** each

In [ ]:
OUTPUT_DIR = "collected_demos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED_START = 100
MAX_ATTEMPT_MULTIPLIER = 3

collection_summary = {}

for scenario_name, cfg in SCENARIOS.items():
    target = cfg["target_demos"]
    max_attempts = target * MAX_ATTEMPT_MULTIPLIER

    print(f"\n{'='*70}")
    print(f"  COLLECTING: {scenario_name}")
    print(f"  Target: {target} successful demos | Max attempts: {max_attempts}")
    print(f"{'='*70}")

    env = suite.make(**cfg["env_kwargs"])
    policy = cfg["policy_class"]()

    hdf5_path = os.path.join(OUTPUT_DIR, f"{scenario_name.lower()}_demos.hdf5")
    h5_file = h5py.File(hdf5_path, "w")
    data_grp = h5_file.create_group("data")
    data_grp.attrs["env"] = scenario_name
    data_grp.attrs["policy"] = "scripted"
    data_grp.attrs["task_description"] = TASK_DESCRIPTIONS[scenario_name]

    successes = 0
    attempts = 0
    start_time = time.time()

    for ep in range(max_attempts):
        if successes >= target:
            break

        np.random.seed(SEED_START + ep)
        result = run_episode(
            env, policy, scenario_name,
            max_steps=cfg["max_steps"],
            record_video=False,
        )
        attempts += 1

        if result["success"]:
            demo_grp = data_grp.create_group(f"demo_{successes}")
            demo_grp.attrs["seed"] = SEED_START + ep
            demo_grp.attrs["num_samples"] = len(result["actions"])
            demo_grp.attrs["total_reward"] = result["total_reward"]

            demo_grp.create_dataset("actions", data=np.array(result["actions"]))

            obs_grp = demo_grp.create_group("obs")
            if result["observations"]:
                for key in result["observations"][0].keys():
                    obs_data = np.array([o[key] for o in result["observations"]])
                    obs_grp.create_dataset(key, data=obs_data)

            h5_file.flush()
            successes += 1

        if attempts % 10 == 0 or successes >= target:
            elapsed = time.time() - start_time
            rate = successes / max(attempts, 1)
            print(f"  [{attempts:4d} attempts]  {successes}/{target} demos  "
                  f"(rate: {rate:.1%})  {elapsed:.0f}s")

    env.close()

    data_grp.attrs["total_demos"] = successes
    data_grp.attrs["total_attempts"] = attempts
    data_grp.attrs["success_rate"] = successes / max(attempts, 1)
    h5_file.close()

    elapsed = time.time() - start_time
    collection_summary[scenario_name] = {
        "successes": successes, "target": target,
        "attempts": attempts, "rate": successes / max(attempts, 1),
        "hdf5_path": hdf5_path, "elapsed_s": elapsed,
    }

    status = "DONE" if successes >= target else "INCOMPLETE"
    print(f"  {status}: {successes}/{target} in {attempts} attempts ({elapsed:.0f}s)")

# Final summary
print(f"\n\n{'='*70}")
print(f"  DATA COLLECTION SUMMARY")
print(f"{'='*70}")
print(f"  {'Scenario':<25s} {'Collected':>10s} {'Attempts':>10s} {'Rate':>8s} {'Time':>8s}")
print(f"  {'-'*65}")
for name, r in collection_summary.items():
    print(f"  {name:<25s} {r['successes']:>5d}/{r['target']:<4d}  "
          f"{r['attempts']:>9d}  {r['rate']:>7.1%}  {r['elapsed_s']:>6.0f}s")

print(f"\n  HDF5 files in: {OUTPUT_DIR}/")
for name, r in collection_summary.items():
    print(f"    {r['hdf5_path']}  ({r['successes']} demos)")

---
## 6. Convert HDF5 to LeRobot v3.0 Format

Convert collected HDF5 demos into the format SmolVLA expects:
- **Parquet** files with per-frame rows (actions, states, episode_index, timestamps)
- **MP4 videos** for each episode (agentview + wrist camera)
- **Metadata** JSON

SmolVLA training expects these exact keys:
- `observation.images.image` -- agentview camera (256x256)
- `observation.images.image2` -- wrist camera (256x256)
- `observation.state` -- [eef_pos(3), eef_quat(4)] = 7D
- `action` -- [dx, dy, dz, dax, day, daz, gripper] = 7D

In [ ]:
import pandas as pd
from PIL import Image as PILImage

LEROBOT_DIR = "lerobot_dataset"

def _build_state_vector(eef_pos, eef_quat, t):
    """Build [eef_pos(3) + eef_quat(4)] = 7-dim state vector."""
    return np.concatenate([eef_pos[t].astype(np.float32),
                           eef_quat[t].astype(np.float32)])

def convert_hdf5_to_lerobot(hdf5_path, scenario_name, output_base_dir):
    """Convert one HDF5 file to LeRobot v3.0 format (local only, no HuggingFace)."""
    import shutil

    output_dir = os.path.join(output_base_dir, scenario_name.lower())
    # Clean previous conversion to avoid stale files
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    chunk_dir = os.path.join(output_dir, "data", "chunk-000")
    os.makedirs(chunk_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "meta"), exist_ok=True)

    task_desc = TASK_DESCRIPTIONS[scenario_name]
    action_names = ["dx", "dy", "dz", "dax", "day", "daz", "gripper"]

    all_rows = []
    episode_lengths = []
    global_idx = 0
    has_agentview = False
    has_wrist = False

    with h5py.File(hdf5_path, "r") as f:
        demo_keys = sorted([k for k in f["data"].keys() if k.startswith("demo_")])
        print(f"  {scenario_name}: {len(demo_keys)} demos")

        for ep_idx, demo_key in enumerate(demo_keys):
            demo = f["data"][demo_key]
            actions = demo["actions"][:]
            n_steps = len(actions)
            episode_lengths.append(n_steps)

            has_agentview = "agentview_image" in demo["obs"]
            has_wrist = "robot0_eye_in_hand_image" in demo["obs"]
            eef_pos = demo["obs"]["robot0_eef_pos"][:] if "robot0_eef_pos" in demo["obs"] else np.zeros((n_steps, 3))
            eef_quat = demo["obs"]["robot0_eef_quat"][:] if "robot0_eef_quat" in demo["obs"] else np.zeros((n_steps, 4))

            # Save agentview video
            if has_agentview:
                vid_dir = os.path.join(output_dir, "videos", "observation.images.image", "chunk-000")
                os.makedirs(vid_dir, exist_ok=True)
                vid_path = os.path.join(vid_dir, f"episode_{ep_idx:06d}.mp4")
                imgs = demo["obs"]["agentview_image"][:]
                writer = imageio.get_writer(vid_path, fps=20)
                for frame in imgs:
                    frame = np.flip(frame, axis=0).copy()
                    if frame.shape[0] != 256:
                        frame = np.array(PILImage.fromarray(frame).resize((256, 256)))
                    writer.append_data(frame)
                writer.close()

            # Save wrist video
            if has_wrist:
                vid_dir = os.path.join(output_dir, "videos", "observation.images.image2", "chunk-000")
                os.makedirs(vid_dir, exist_ok=True)
                vid_path = os.path.join(vid_dir, f"episode_{ep_idx:06d}.mp4")
                imgs = demo["obs"]["robot0_eye_in_hand_image"][:]
                writer = imageio.get_writer(vid_path, fps=20)
                for frame in imgs:
                    frame = np.flip(frame, axis=0).copy()
                    if frame.shape[0] != 256:
                        frame = np.array(PILImage.fromarray(frame).resize((256, 256)))
                    writer.append_data(frame)
                writer.close()

            # Build rows with array columns
            for t in range(n_steps):
                row = {
                    "index": global_idx,
                    "episode_index": ep_idx,
                    "frame_index": t,
                    "timestamp": t / 20.0,
                    "task_index": 0,
                    "action": actions[t].astype(np.float32).tolist(),
                    "observation.state": _build_state_vector(eef_pos, eef_quat, t).tolist(),
                }
                all_rows.append(row)
                global_idx += 1

    # Save data parquet
    df = pd.DataFrame(all_rows)
    parquet_path = os.path.join(chunk_dir, "file-000.parquet")
    df.to_parquet(parquet_path, index=False)

    # Build features dict
    features = {
        "action": {
            "dtype": "float32",
            "shape": [7],
            "names": {"motors": action_names},
        },
        "observation.state": {
            "dtype": "float32",
            "shape": [7],
            "names": {"motors": ["eef_x", "eef_y", "eef_z",
                                  "quat_x", "quat_y", "quat_z", "quat_w"]},
        },
        "timestamp": {"dtype": "float32", "shape": [1], "names": None},
        "frame_index": {"dtype": "int64", "shape": [1], "names": None},
        "episode_index": {"dtype": "int64", "shape": [1], "names": None},
        "index": {"dtype": "int64", "shape": [1], "names": None},
        "task_index": {"dtype": "int64", "shape": [1], "names": None},
    }

    if has_agentview:
        features["observation.images.image"] = {
            "dtype": "video", "shape": [256, 256, 3],
            "names": ["height", "width", "channel"],
            "video_info": {"video.fps": 20.0, "video.codec": "libx264",
                           "video.pix_fmt": "yuv420p", "video.is_depth_map": False,
                           "has_audio": False},
        }

    if has_wrist:
        features["observation.images.image2"] = {
            "dtype": "video", "shape": [256, 256, 3],
            "names": ["height", "width", "channel"],
            "video_info": {"video.fps": 20.0, "video.codec": "libx264",
                           "video.pix_fmt": "yuv420p", "video.is_depth_map": False,
                           "has_audio": False},
        }

    n_eps = len(demo_keys)
    meta = {
        "codebase_version": "v3.0",
        "robot_type": "panda",
        "total_episodes": n_eps,
        "total_frames": sum(episode_lengths),
        "total_tasks": 1,
        "total_videos": n_eps * (int(has_agentview) + int(has_wrist)),
        "total_chunks": 1,
        "chunks_size": 1000,
        "fps": 20,
        "splits": {"train": f"0:{n_eps}"},
        "data_path": "data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet",
        "video_path": "videos/{video_key}/chunk-{chunk_index:03d}/episode_{episode_index:06d}.mp4",
        "features": features,
    }

    with open(os.path.join(output_dir, "meta", "info.json"), "w") as f:
        json.dump(meta, f, indent=2)

    # Save episodes as single parquet file (LeRobot expects meta/episodes.parquet)
    episodes_records = []
    for i, length in enumerate(episode_lengths):
        episodes_records.append({
            "episode_index": i,
            "tasks": json.dumps([task_desc]),
            "length": length,
        })
    episodes_path = os.path.join(output_dir, "meta", "episodes.parquet")
    pd.DataFrame(episodes_records).to_parquet(episodes_path, index=False)

    # Save tasks as single parquet file (LeRobot expects meta/tasks.parquet)
    tasks_records = [{"task_index": 0, "task": task_desc}]
    tasks_path = os.path.join(output_dir, "meta", "tasks.parquet")
    pd.DataFrame(tasks_records).to_parquet(tasks_path, index=False)

    print(f"    Saved: {len(df)} rows, {n_eps} episodes")
    print(f"    Parquet: {parquet_path}")
    print(f"    Episodes: {episodes_path}")
    print(f"    Tasks: {tasks_path}")
    print(f"    Videos: {output_dir}/videos/")
    return output_dir


# Convert all scenarios
print(f"\n{'='*60}")
print(f"  Converting HDF5 -> LeRobot v3.0 Format")
print(f"{'='*60}")

lerobot_paths = {}
for scenario_name, r in collection_summary.items():
    if r["successes"] > 0:
        lerobot_paths[scenario_name] = convert_hdf5_to_lerobot(
            r["hdf5_path"], scenario_name, LEROBOT_DIR
        )
    else:
        print(f"  {scenario_name}: SKIPPED (0 demos)")

print(f"\nConversion complete. LeRobot datasets in: {LEROBOT_DIR}/")

---
## 7. Train SmolVLA

Fine-tune SmolVLA on the collected demonstrations using `lerobot-train`.

**Estimated time:** ~3-4h on A100, ~6-8h on T4

You can train on individual scenarios or combine all data.

In [ ]:
# Auto-detect GPU and set batch size
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 4 if gpu_mem_gb > 30 else 2
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.0f}GB) -> batch_size={BATCH_SIZE}")
else:
    BATCH_SIZE = 1
    print("WARNING: No GPU detected")

# Pick which scenario to train on (change as needed)
TRAIN_SCENARIO = "Lift"  # Start with simplest
TRAIN_STEPS = 20000
CHECKPOINT_DIR = f"outputs/checkpoints/smolvla_{TRAIN_SCENARIO.lower()}"

train_dataset = lerobot_paths.get(TRAIN_SCENARIO)
if train_dataset:
    print(f"\nTraining SmolVLA on: {TRAIN_SCENARIO}")
    print(f"Dataset: {train_dataset}")
    print(f"Steps: {TRAIN_STEPS}, Batch: {BATCH_SIZE}")
    print(f"Output: {CHECKPOINT_DIR}")
else:
    print(f"No dataset for {TRAIN_SCENARIO}. Run collection first.")

In [ ]:
# Launch training (LOCAL dataset only)
# Dataset is loaded from local files (meta/tasks.parquet, meta/episodes.parquet, etc.)
# VLM weights are downloaded from HuggingFace on first run and cached.
if train_dataset:
    !lerobot-train \
        --policy.type=smolvla \
        --policy.repo_id=local/smolvla_{TRAIN_SCENARIO.lower()} \
        --policy.load_vlm_weights=true \
        --policy.push_to_hub=false \
        --dataset.repo_id={TRAIN_SCENARIO.lower()} \
        --dataset.root={LEROBOT_DIR}/{TRAIN_SCENARIO.lower()} \
        --batch_size={BATCH_SIZE} \
        --steps={TRAIN_STEPS} \
        --output_dir={CHECKPOINT_DIR} \
        --save_freq=2000 \
        --eval_freq=2000 \
        --log_freq=100 \
        --seed=42 \
        --policy.device=cuda

In [ ]:
# Post-training: fix n_action_steps for fast inference
import glob

config_files = glob.glob(os.path.join(CHECKPOINT_DIR, "**/config.json"), recursive=True)
for cfg_path in config_files:
    with open(cfg_path) as f:
        cfg = json.load(f)
    if cfg.get("n_action_steps") != 50:
        cfg["n_action_steps"] = 50
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Fixed n_action_steps=50 in {cfg_path}")
    else:
        print(f"Already correct: {cfg_path}")

---
## 8. Evaluate Trained VLA on All 6 Environments

Test the trained SmolVLA model on all 6 scenarios.
Every episode produces:
- **Video** (MP4) showing the robot's behavior
- **SUCCESS / FAIL** label (via `env._check_success()`)
- Per-scenario success rate summary table

In [ ]:
# ============================================================
# VLA EVALUATION HELPERS
# ============================================================
# Uses LeRobot's preprocessor/postprocessor pipeline for correct:
#   - Language tokenization (task string → token IDs)
#   - State normalization (MEAN_STD using dataset statistics)
#   - Action unnormalization (MEAN_STD using dataset statistics)
# ============================================================

def load_vla_policy(checkpoint_path, device="cuda"):
    """Load trained SmolVLA policy with preprocessor/postprocessor pipelines.
    
    Returns (policy, preprocess, postprocess) tuple.
    The preprocessor handles language tokenization and state normalization.
    The postprocessor handles action unnormalization.
    """
    try:
        from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
        from lerobot.policies.factory import make_pre_post_processors

        policy = SmolVLAPolicy.from_pretrained(checkpoint_path)
        policy.to(device)
        policy.eval()

        # Load preprocessor/postprocessor from saved model configs.
        # These contain the dataset normalization stats (mean/std for state & action)
        # and the tokenizer config for language instructions.
        preprocess, postprocess = make_pre_post_processors(
            policy.config,
            checkpoint_path,
            preprocessor_overrides={"device_processor": {"device": str(device)}},
        )

        print(f"Loaded VLA from {checkpoint_path}")
        print(f"  Preprocessor steps: {[type(s).__name__ for s in preprocess.steps]}")
        print(f"  Postprocessor steps: {[type(s).__name__ for s in postprocess.steps]}")
        return policy, preprocess, postprocess
    except Exception as e:
        import traceback
        print(f"Failed to load VLA: {e}")
        traceback.print_exc()
        print("Using random policy for testing.")
        return None, None, None


def get_vla_action(policy, preprocess, postprocess, obs, task_language, device="cuda"):
    """Get action from SmolVLA given robosuite observation.

    Pipeline:
      1. Build raw observation dict with LeRobot-compatible keys
      2. Convert to tensors, normalize images to [0,1], add batch dim
      3. Preprocess: tokenize language, normalize state (MEAN_STD)
      4. Model inference via select_action
      5. Postprocess: unnormalize action (MEAN_STD)
    """
    if policy is None:
        return np.random.uniform(-0.3, 0.3, size=7)

    from lerobot.policies.utils import prepare_observation_for_inference

    # --- 1. Prepare raw numpy observations with correct LeRobot keys ---

    # Camera 1: agentview (flip vertical for MuJoCo → standard orientation)
    agentview = np.flip(
        obs.get("agentview_image", np.zeros((256, 256, 3), dtype=np.uint8)), axis=0
    ).copy()
    if agentview.shape[0] != 256:
        agentview = np.array(PILImage.fromarray(agentview).resize((256, 256)))

    # Camera 2: wrist camera
    wrist = obs.get("robot0_eye_in_hand_image", None)
    if wrist is not None:
        wrist = np.flip(wrist, axis=0).copy()
        if wrist.shape[0] != 256:
            wrist = np.array(PILImage.fromarray(wrist).resize((256, 256)))
    else:
        wrist = np.zeros((256, 256, 3), dtype=np.uint8)

    # State: eef_pos(3) + eef_quat(4) = 7-dim
    eef_pos = obs.get("robot0_eef_pos", np.zeros(3))
    eef_quat = obs.get("robot0_eef_quat", np.zeros(4))
    state = np.concatenate([eef_pos, eef_quat]).astype(np.float32)

    # Raw observation dict with LeRobot keys (numpy arrays)
    raw_obs = {
        "observation.images.image": agentview,
        "observation.images.image2": wrist,
        "observation.state": state,
    }

    # --- 2. Convert to tensors + normalize images + add batch dim ---
    # prepare_observation_for_inference:
    #   - Converts numpy → torch tensors
    #   - Images: permute (H,W,C)→(C,H,W), normalize to [0,1], add batch dim
    #   - State: add batch dim
    #   - Adds "task" string to dict
    obs_frame = prepare_observation_for_inference(raw_obs, device, task=task_language)

    # --- 3. Preprocess: tokenize language, normalize state ---
    # The preprocessor pipeline:
    #   - SmolVLANewLineProcessor: ensures task ends with \n
    #   - TokenizerProcessorStep: task string → observation.language.tokens + attention_mask
    #   - NormalizerProcessorStep: MEAN_STD normalize observation.state using dataset stats
    obs_preprocessed = preprocess(obs_frame)

    # --- 4. Model inference ---
    with torch.no_grad():
        action = policy.select_action(obs_preprocessed)

    # --- 5. Postprocess: unnormalize action ---
    # UnnormalizerProcessorStep: MEAN_STD unnormalize action using dataset stats
    action = postprocess(action)

    if isinstance(action, torch.Tensor):
        action = action.cpu().numpy().flatten()

    return np.clip(action[:7], -1, 1)


def run_vla_episode(env, vla_policy, preprocess, postprocess, scenario_name,
                    task_language, max_steps, device="cuda", record_video=True):
    """Run one evaluation episode with VLA policy.
    Returns: success (bool), total_reward, steps, frames.
    """
    obs = env.reset()
    if vla_policy is not None:
        vla_policy.reset()  # clear internal action queue

    frames = []
    total_reward = 0.0
    success = False

    for step in range(max_steps):
        action = get_vla_action(vla_policy, preprocess, postprocess,
                                obs, task_language, device=device)

        if record_video and "agentview_image" in obs:
            frames.append(np.flip(obs["agentview_image"], axis=0).copy())

        obs, reward, done, info = env.step(action)
        total_reward += reward

        if env._check_success():
            success = True

        if done:
            break

    if record_video and "agentview_image" in obs:
        frames.append(np.flip(obs["agentview_image"], axis=0).copy())

    return success, total_reward, step + 1, frames


print("VLA evaluation helpers loaded (with preprocessor/postprocessor pipeline).")

In [ ]:
# ============================================================
# RUN VLA EVALUATION ON ALL 6 SCENARIOS
# ============================================================

# Re-define paths in case training cell was skipped
TRAIN_SCENARIO = "Lift"
CHECKPOINT_DIR = f"outputs/checkpoints/smolvla_{TRAIN_SCENARIO.lower()}"

EVAL_EPISODES = 10   # episodes per scenario for evaluation
EVAL_VIDEO_DIR = "eval_videos"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load trained policy WITH preprocessor/postprocessor
vla_policy, vla_preprocess, vla_postprocess = load_vla_policy(CHECKPOINT_DIR, device=DEVICE)

eval_results = {}

for scenario_name, cfg in SCENARIOS.items():
    task_lang = TASK_DESCRIPTIONS[scenario_name]
    max_steps = cfg["max_steps"]

    print(f"\n{'='*70}")
    print(f"  EVALUATING VLA: {scenario_name}")
    print(f"  Task: {task_lang}")
    print(f"  Episodes: {EVAL_EPISODES}, Max steps: {max_steps}")
    print(f"{'='*70}")

    env = suite.make(**cfg["env_kwargs"])
    ep_results = []

    for ep in range(EVAL_EPISODES):
        np.random.seed(1000 + ep)  # different seeds from collection

        success, reward, steps, frames = run_vla_episode(
            env, vla_policy, vla_preprocess, vla_postprocess,
            scenario_name, task_lang,
            max_steps=max_steps, device=DEVICE, record_video=True,
        )

        status = "SUCCESS" if success else "FAIL"

        ep_results.append({
            "episode": ep, "success": success,
            "reward": reward, "steps": steps,
        })

        print(f"  Episode {ep+1:2d}: {status:7s}  "
              f"reward={reward:.2f}  steps={steps}")

        # Save video with success/fail in filename
        tag = "success" if success else "fail"
        video_path = os.path.join(
            EVAL_VIDEO_DIR, scenario_name,
            f"ep{ep+1:02d}_{tag}.mp4"
        )
        if frames:
            save_video_file(frames, video_path)

    env.close()

    n_success = sum(1 for r in ep_results if r["success"])
    rate = n_success / EVAL_EPISODES

    eval_results[scenario_name] = {
        "n_success": n_success,
        "n_episodes": EVAL_EPISODES,
        "success_rate": rate,
        "episodes": ep_results,
    }

    print(f"\n  {scenario_name} RESULT: {n_success}/{EVAL_EPISODES} ({rate:.0%})")

# ============================================================
# EVALUATION SUMMARY TABLE
# ============================================================
print(f"\n\n{'='*70}")
print(f"  VLA EVALUATION SUMMARY")
print(f"{'='*70}")
print(f"  {'Scenario':<25s} {'Success':>10s} {'Rate':>8s} {'Status':>10s}")
print(f"  {'-'*55}")

total_s, total_e = 0, 0
for name, r in eval_results.items():
    status = "PASS" if r["n_success"] > 0 else "FAIL"
    print(f"  {name:<25s} {r['n_success']:>3d}/{r['n_episodes']:<3d}  "
          f"{r['success_rate']:>7.0%} {status:>10s}")
    total_s += r["n_success"]
    total_e += r["n_episodes"]

print(f"  {'-'*55}")
print(f"  {'OVERALL':<25s} {total_s:>3d}/{total_e:<3d}  {total_s/total_e:>7.0%}")

# Save results JSON
os.makedirs("eval_results", exist_ok=True)
with open("eval_results/vla_eval.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print(f"\n  Results saved to eval_results/vla_eval.json")

In [ ]:
# ============================================================
# DISPLAY EVALUATION VIDEOS INLINE
# ============================================================
from pathlib import Path

for scenario_name in SCENARIOS.keys():
    video_dir = Path(EVAL_VIDEO_DIR) / scenario_name
    if not video_dir.exists():
        continue

    videos = sorted(video_dir.glob("*.mp4"))
    if not videos:
        continue

    r = eval_results[scenario_name]
    display(HTML(
        f'<h3>{scenario_name} -- '
        f'{r["n_success"]}/{r["n_episodes"]} ({r["success_rate"]:.0%})</h3>'
    ))

    for video_path in videos:
        name = video_path.stem
        is_success = "success" in name
        color = "green" if is_success else "red"
        status = "SUCCESS" if is_success else "FAIL"

        display(HTML(
            f'<span style="color: {color}; font-weight: bold;">'
            f'{name} -- {status}</span>'
        ))
        show_video_inline(str(video_path))

In [ ]:
# ============================================================
# SUCCESS/FAIL GRID VISUALIZATION
# ============================================================
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(eval_results), figsize=(3 * len(eval_results), 4))
if len(eval_results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, eval_results.items()):
    episodes = r["episodes"]
    cols = min(5, len(episodes))
    rows = (len(episodes) + cols - 1) // cols

    for i, ep in enumerate(episodes):
        row = i // cols
        col = i % cols
        color = '#4CAF50' if ep['success'] else '#f44336'
        rect = mpatches.FancyBboxPatch(
            (col, rows - 1 - row), 0.9, 0.9,
            boxstyle="round,pad=0.05", facecolor=color,
            edgecolor='white', linewidth=2
        )
        ax.add_patch(rect)
        ax.text(col + 0.45, rows - 1 - row + 0.45, str(i + 1),
                ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    ax.set_xlim(-0.1, cols + 0.1)
    ax.set_ylim(-0.1, rows + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f"{name}\n{r['success_rate']:.0%}", fontsize=10, fontweight='bold')

success_patch = mpatches.Patch(color='#4CAF50', label='Success')
fail_patch = mpatches.Patch(color='#f44336', label='Fail')
fig.legend(handles=[success_patch, fail_patch], loc='lower center', ncol=2, fontsize=11)

plt.suptitle(f"VLA Evaluation: {total_s}/{total_e} ({total_s/total_e:.0%}) Overall",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("eval_results/eval_grid.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 9. (Optional) Save to Google Drive

In [ ]:
# Uncomment to save everything to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# drive_dir = "/content/drive/MyDrive/robosuite_vla"
# os.makedirs(drive_dir, exist_ok=True)
#
# # Copy demos, dataset, results, videos
# for src_dir in ["collected_demos", "lerobot_dataset", "eval_results",
#                 "eval_videos", "trial_videos"]:
#     if os.path.exists(src_dir):
#         shutil.copytree(src_dir, os.path.join(drive_dir, src_dir), dirs_exist_ok=True)
#
# # Copy checkpoint
# if os.path.exists(CHECKPOINT_DIR):
#     shutil.copytree(CHECKPOINT_DIR, os.path.join(drive_dir, "checkpoint"), dirs_exist_ok=True)
#
# print(f"All saved to {drive_dir}")